# Airbnb Paris – Exp 1: AnoLLM
- LoRA-Finetuning Qwen2.5-0.5B auf serialisierten Zeilen, Score = NLL
- Exp 1 = unsupervised: Training auf vollem Train ohne Labels; lesbare Rohwerte + Freitexte

In [1]:
import sys, time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
sys.path.insert(0, "../../anollm_src")
from anollm.anollm import AnoLLM

/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Daten laden
- Rohdaten einlesen (Aufbereitung folgt in der nächsten Zelle)

In [ ]:
raw = pd.read_csv("../../data/raw/airbnb_paris.csv", low_memory=False)
print("geladen:", raw.shape)

## Spalten droppen & aufbereiten
- ICC-Filter + Label; Leakage-/ID-/Meta-Spalten droppen; lesbare Werte (%-Raten, Counts), NaN behandeln, Split

In [ ]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]

raw = raw.dropna(subset=["review_scores_rating"])
raw = raw[(raw["review_scores_rating"] == 5.0) | (raw["review_scores_rating"] <= 3.0)].copy()
y = (raw["review_scores_rating"] != 5.0).astype(int).values  # Outlier = rating <= 3

# Leakage (review_scores/-counts/-daten) + IDs/URLs/Meta/100%-NaN/Redundanzen entfernen
drop = ["id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url", "host_id", "host_url",
        "host_name", "host_thumbnail_url", "host_picture_url", "license",
        "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin",
        "review_scores_communication", "review_scores_location", "review_scores_value",
        "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
        "first_review", "last_review", "reviews_per_month", "calendar_updated", "calendar_last_scraped",
        "host_listings_count", "host_total_listings_count", "minimum_minimum_nights", "maximum_minimum_nights",
        "minimum_maximum_nights", "maximum_maximum_nights", "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",
        "has_availability", "host_neighbourhood", "neighbourhood", "price", "beds", "bathrooms",
        "estimated_revenue_l365d", "neighbourhood_group_cleansed"]
raw = raw.drop(columns=[c for c in drop if c in raw.columns])

# lesbare Werte: %-Raten als Zahl, Listen als Count; Rest roh (Datum/Ort/Kategorien als Strings)
raw["host_response_rate"] = pd.to_numeric(raw["host_response_rate"].astype(str).str.rstrip("%"), errors="coerce")
raw["host_acceptance_rate"] = pd.to_numeric(raw["host_acceptance_rate"].astype(str).str.rstrip("%"), errors="coerce")
raw["amenities_count"] = raw["amenities"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
raw["host_verifications_count"] = raw["host_verifications"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
feat = raw.drop(columns=["amenities", "host_verifications"])

num_cols = [c for c in feat.columns if pd.api.types.is_numeric_dtype(feat[c])]
str_cols = [c for c in feat.columns if c not in num_cols + text_cols]
feat[str_cols] = feat[str_cols].fillna("missing")
feat[text_cols] = feat[text_cols].fillna("")
feat[num_cols] = feat[num_cols].fillna(feat[num_cols].median())

idx = np.arange(len(feat))
tr, te = train_test_split(idx, test_size=0.3, stratify=y, random_state=42)
df_train = feat.iloc[tr].reset_index(drop=True)
df_test = feat.iloc[te].reset_index(drop=True)
y_test = y[te]
print("train", df_train.shape, "test", df_test.shape, "test outlier rate", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_1")

## AnoLLM trainieren & Scores (NLL)
- `max_length_dict` begrenzt Textspalten (Token-Budget); Vorzeichen-Auto-Korrektur
- Gradient Checkpointing: senkt Trainingsspeicher (Airbnb hat längere Zeilen), Modell bleibt identisch

## Single-GPU-Setup (kein DDP/NCCL)
- Verteilte Env-Variablen entfernen → HF Trainer wrappt nicht in DistributedDataParallel

In [3]:
import os
import torch.distributed as dist
import anollm.anollm_trainer
from torch.utils.data import DataLoader, SequentialSampler

# 1. Alle Cluster-Variablen restlos löschen
for key in ["LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(key, None)

# 2. Falls noch eine alte Prozessgruppe aktiv ist, sauber beenden
if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()

# 3. MONKEY-PATCH: Wir definieren einen sauberen Single-GPU Dataloader
def single_gpu_get_train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self._train_batch_size,
        sampler=SequentialSampler(self.train_dataset),
        collate_fn=self.data_collator,
        drop_last=True
    )

# 4. Die fehlerhafte Funktion der Amazon-Bibliothek im RAM überschreiben
anollm.anollm_trainer.AnoLLMTrainer.get_train_dataloader = single_gpu_get_train_dataloader

print("🚀 Single-GPU-Modus aktiv, Trainer gepatcht!")

🚀 Single-GPU-Modus aktiv, Trainer gepatcht!


In [4]:
max_len = {c: 64 for c in text_cols}
model = AnoLLM(llm="Qwen/Qwen2.5-0.5B", efficient_finetuning="lora", textual_columns=text_cols,
               max_length_dict=max_len, batch_size=2, max_steps=2000, learning_rate=5e-4,
               gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
model.model.enable_input_require_grads()  # nötig für Gradient Checkpointing mit LoRA/PEFT

t0 = time.perf_counter()
model.fit(df_train)
scores = model.decision_function(df_test, n_permutations=8, batch_size=2, device="cuda").mean(axis=1)
runtime = time.perf_counter() - t0

scores = np.asarray(scores).astype(float)
auc = roc_auc_score(y_test, scores)
if auc < 0.5:
    scores = -scores
    auc = roc_auc_score(y_test, scores)
ap = average_precision_score(y_test, scores)

with mlflow.start_run(run_name="anollm"):
    mlflow.log_params({"llm": "Qwen/Qwen2.5-0.5B", "max_steps": 2000, "n_permutations": 8})
    mlflow.log_metric("average_precision", ap)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("runtime_s", runtime)
print(f"anollm: AP={ap:.4f} AUC={auc:.4f} time={runtime:.1f}s")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4969.68it/s]


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


100%|██████████| 12845/12845 [01:29<00:00, 143.74it/s]


Preprocessing done.
Data 0:  number_of_reviews is -0.1524715289477803, calculated_host_listings_count_shared_rooms is -0.0459090318505475, host_location_missing is 0, host_response_time_unknown is 0.0, estimated_occupancy_l365d is 0.179113880914602, room_type_Shared room is 0.0, has_availability is 1, host_response_rate is 0.3258507505005285, minimum_minimum_nights is -0.3657878214500038, reviews_per_month is 0.6581167246144466, host_response_time_within a few hours is 0.0, amenities_count is -0.536809531906397, host_is_superhost is 0, host_in_paris is 1, number_of_reviews_ltm is 0.3952017612601827, host_response_time_a few days or more is 0.0, host_total_listings_count is -0.2475233460544459, neighborhood_overview is None, minimum_maximum_nights is -0.9818343235207242, property_type is 0.4304614778762278, instant_bookable is 0, availability_90 is -0.7258677748346803, host_in_france is 1, number_of_reviews_ly is -0.452033498380893, room_type_Entire home/apt is 1.0, maximum_maximum_nigh

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
500,0.797738
1000,0.573327
1500,0.525067
2000,0.490865


100%|██████████| 5505/5505 [00:36<00:00, 149.09it/s]


Preprocessing done.


100%|██████████| 8/8 [1:52:43<00:00, 845.42s/it]

anollm: AP=0.0709 AUC=0.6429 time=7817.4s
